# Bước 00: Hotfix Join Thời Tiết Causal — Notebook Kiểm Chứng Độc Lập
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

## 1. TỔNG QUAN VÀ MỤC TIÊU

Notebook này **tách riêng** đoạn hotfix join thời tiết causal (vốn nằm trong
`01_reindex_mask_outlier.ipynb`, bước 3.1) ra một file độc lập, để kiểm chứng/trình bày
tách biệt mà **không đụng vào pipeline chính đang chạy** (01→06).

**An toàn tuyệt đối với pipeline đang chạy:** notebook này chỉ ĐỌC
`data/mlmart_base/v3_preprocessing.parquet` (input gốc, không phải output của bất kỳ bước
nào trong 01-06), và GHI ra thư mục riêng biệt hoàn toàn
`data/model/v3/00_hotfix_audit/` — không trùng, không ghi đè bất kỳ file nào pipeline
chính (01→06) đang đọc hoặc ghi.

**Bối cảnh bug:** trước hotfix (2026-07-30), một số dòng bị join với `weather_timestamp`
**SAU** `timestamp` của chính nó — tức là dùng dữ liệu thời tiết CHƯA XẢY RA tại thời điểm
dự báo (leakage thời gian). Hotfix join lại theo quy tắc causal: mỗi dòng chỉ được lấy thời
tiết của khung giờ đã trôi qua (`floor` về đầu giờ), không bao giờ lấy giờ tương lai.

## 2. Import thư viện và khai báo tham số

In [ ]:
import gc
import os

import numpy as np
import pandas as pd

# ── Duong dan: CHI doc file goc, CHI ghi ra thu muc rieng, khong dung path nao
# cua pipeline chinh (01->06) dang chay song song. ──
INPUT_PATH = '../../data/mlmart_base/v3_preprocessing.parquet'
OUTPUT_DIR = '../../data/model/v3/00_hotfix_audit'
OUTPUT_PATH = f'{OUTPUT_DIR}/v3_preprocessing_causal_hotfix_audit.parquet'

SITE_COL = 'site_id'
TIMESTAMP_COL = 'timestamp'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Đã import thư viện và khai báo tham số.")
print(f"- Đọc từ : {INPUT_PATH}")
print(f"- Ghi ra : {OUTPUT_PATH}")

## 3. Đọc dữ liệu gốc

In [ ]:
df = pd.read_parquet(INPUT_PATH)
print(f"Đang đọc dữ liệu từ: {INPUT_PATH}")
print(f"Tổng số dòng: {len(df)}")
print(f"Số site: {df[SITE_COL].nunique()}")
display(df.head(3))

## 4. Đo lường TRƯỚC khi sửa — bằng chứng bug leakage thời gian

In [ ]:
COT_THOI_TIET = [
    'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
    'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid',
    'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration',
    'weather_code', 'weather_is_day', 'weather_type_is_day',
    'weather_condition', 'weather_description', 'weather_id', 'weather_type_id',
]
_tt = [c for c in COT_THOI_TIET if c in df.columns]

df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL])
df['weather_timestamp'] = pd.to_datetime(df['weather_timestamp'])

_delta_truoc = (df['weather_timestamp'] - df[TIMESTAMP_COL]).dt.total_seconds() / 60
_leak_truoc = int((_delta_truoc > 0).sum())
print("--- TRƯỚC KHI SỬA ---")
print(f"Dòng dùng thời tiết TƯƠNG LAI: {_leak_truoc:,}/{len(df):,} ({_leak_truoc / len(df) * 100:.2f}%)")
print("Phân bố (weather_timestamp - timestamp) theo phút:")
print(_delta_truoc.value_counts().sort_index().to_string())

### Nhận xét
Nếu `_leak_truoc > 0`, nghĩa là dữ liệu gốc vẫn còn dòng dùng thời tiết chưa xảy ra tại
thời điểm dự báo — đây chính xác là bug leakage thời gian gây trễ pha đã fix ngày
2026-07-30. Nếu `_leak_truoc == 0`, nghĩa là bug đã được xử lý ở nguồn
(`v3_preprocessing.parquet`) từ trước, và bước dưới đây chạy như lưới an toàn (idempotent).

## 5. Áp dụng hotfix — join lại thời tiết theo đúng quy tắc causal

In [ ]:
# Bang tra thoi tiet: moi (site, nhan gio) mot ban ghi, lay tu chinh du lieu dang co
_bang_tt = (df[[SITE_COL, 'weather_timestamp'] + _tt]
            .dropna(subset=['weather_timestamp'])
            .drop_duplicates([SITE_COL, 'weather_timestamp'])
            .rename(columns={'weather_timestamp': '_nhan_gio'}))
print(f"Bảng tra thời tiết: {len(_bang_tt):,} bản ghi (site x nhãn giờ)")

# Nhan gio HOP LE cho moi dong = floor ve dau gio (thoi tiet do da co san tai thoi diem do)
df['_nhan_gio'] = df[TIMESTAMP_COL].dt.floor('h')
_truoc = df[_tt].copy()
df = df.drop(columns=_tt).merge(_bang_tt, on=[SITE_COL, '_nhan_gio'], how='left')

print("Đã join lại thời tiết theo quy tắc causal (floor về đầu giờ đã trôi qua).")

## 6. Đo lường SAU khi sửa — xác nhận hết leakage

In [ ]:
_khop = df[_tt].notna().all(axis=1)
print("--- SAU KHI SỬA ---")
print(f"Dòng join được thời tiết: {int(_khop.sum()):,}/{len(df):,} ({_khop.mean() * 100:.2f}%)")
_delta_sau = (df['_nhan_gio'] - df[TIMESTAMP_COL]).dt.total_seconds() / 60
print(f"Dòng dùng thời tiết TƯƠNG LAI: {int((_delta_sau > 0).sum()):,} (phải bằng 0)")
print(f"Phân bố delta mới (phút): {sorted(_delta_sau.dropna().unique().tolist())}")

_bang_doi = []
for c in _tt:
    if pd.api.types.is_numeric_dtype(_truoc[c]):
        _kh = ~np.isclose(_truoc[c].to_numpy(dtype='float64'),
                          df[c].to_numpy(dtype='float64'), equal_nan=True)
    else:
        _kh = (_truoc[c].astype(str).to_numpy() != df[c].astype(str).to_numpy())
    _bang_doi.append({'cot': c, 'so_dong_doi': int(_kh.sum()),
                      'ty_le_%': round(_kh.mean() * 100, 1)})
print("\nSố dòng thực sự đổi giá trị sau khi sửa join:")
display(pd.DataFrame(_bang_doi).sort_values('so_dong_doi', ascending=False))

## 7. Cổng kiểm tra — dừng nếu vẫn còn leakage

In [ ]:
df['weather_timestamp'] = df['_nhan_gio']
if 'weather_join_method' in df.columns:
    df.loc[_khop, 'weather_join_method'] = 'hour_causal_floor'
    df.loc[~_khop, 'weather_join_method'] = 'missing_weather'
df = df.drop(columns=['_nhan_gio'])
del _bang_tt, _truoc
gc.collect()

assert int((_delta_sau > 0).sum()) == 0, "CONG KIEM TRA KHONG DAT: Van con dong dung thoi tiet tuong lai"
print("CỔNG KIỂM TRA: ĐẠT — không còn dòng nào dùng dữ liệu thời tiết tương lai.")

## 8. Export Processed Dataset — ghi ra thư mục audit riêng

In [ ]:
df.to_parquet(OUTPUT_PATH, index=False)
print("--- HOÀN TẤT ---")
print(f"Đã ghi file kiểm chứng ra: {OUTPUT_PATH}")
print(f"Shape cuối cùng: {df.shape[0]} dòng x {df.shape[1]} cột")
print("\nLưu ý: file này CHỈ dùng để kiểm chứng/trình bày, KHÔNG được pipeline chính")
print("(01->06) đọc hay ghi đè. Không ảnh hưởng tới lần train đang chạy.")